# Train Contract Clarity LoRA

Use this notebook with the Google Colab VS Code extension or directly in Google Colab.

Before running:
1. Set the runtime to GPU, preferably T4.
2. Upload `training/lora_train.jsonl` when prompted.
3. Optionally upload `training/lora_eval.jsonl` for evaluation holdout tracking.

The output adapter will be saved to `models/contract-clarity-lora` and zipped as `contract-clarity-lora.zip`.

In [ ]:
!nvidia-smi

## Install Dependencies

This installs the training stack needed for QLoRA/PEFT.

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers datasets peft accelerate bitsandbytes

## Upload Training Files

Upload these local files when prompted:

- `training/lora_train.jsonl`
- `training/lora_eval.jsonl` optional

If you only upload `lora_train.jsonl`, training still works.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

Path("training").mkdir(exist_ok=True)
uploaded = files.upload()

for name in uploaded:
    destination = Path("training") / Path(name).name
    shutil.move(name, destination)
    print(f"Saved {destination}")

## Validate Dataset Files

In [ ]:
import json
from pathlib import Path

REQUIRED_KEYS = {"instruction", "context", "response"}

def validate_jsonl(path):
    path = Path(path)
    count = 0
    errors = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            count += 1
            row = json.loads(line)
            missing = REQUIRED_KEYS - set(row)
            if missing:
                errors.append(f"Line {line_number}: missing {sorted(missing)}")
            if "## Supporting Citations" not in row.get("response", ""):
                errors.append(f"Line {line_number}: missing Supporting Citations")
            if "[Chunk 1]" not in row.get("response", ""):
                errors.append(f"Line {line_number}: missing [Chunk 1]")
    if errors:
        raise ValueError("\n".join(errors[:20]))
    print(f"Validated {count} examples in {path}")

validate_jsonl("training/lora_train.jsonl")
if Path("training/lora_eval.jsonl").exists():
    validate_jsonl("training/lora_eval.jsonl")

## Training Configuration

Start with `Qwen/Qwen2.5-7B-Instruct`. If Colab runs out of memory, switch to `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # fallback if 7B is too large

DATASET_PATH = "training/lora_train.jsonl"
OUTPUT_DIR = "models/contract-clarity-lora"
EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_LENGTH = 3072

## Train PEFT LoRA Adapter

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

def format_example(example):
    return (
        "<|system|>\n"
        "You are a legal document assistant. Explain contract language in plain English, "
        "use only retrieved context, cite chunks, and avoid legal advice.\n"
        "<|user|>\n"
        f"Instruction:\n{example['instruction']}\n\n"
        f"Retrieved Context:\n{example['context']}\n"
        "<|assistant|>\n"
        f"{example['response']}"
    )

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=quantization_config,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize(example):
    text = format_example(example)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

## Quick Smoke Test

In [ ]:
from peft import PeftModel
from transformers import pipeline

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=quantization_config,
)
adapter_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
pipe = pipeline("text-generation", model=adapter_model, tokenizer=tokenizer, max_new_tokens=350, do_sample=False)

prompt = """<|system|>
You are a legal document assistant. Explain contract language in plain English, use only retrieved context, cite chunks, and avoid legal advice.
<|user|>
Instruction:
Explain the payment clause in plain English using only the retrieved context.

Retrieved Context:
[Chunk 1]
Tenant must pay monthly rent of $2,100 on the first day of each month. Late payments incur a $75 late fee.
<|assistant|>
"""

print(pipe(prompt)[0]["generated_text"].split("<|assistant|>")[-1].strip())

## Zip And Download Adapter

In [ ]:
!zip -r contract-clarity-lora.zip models/contract-clarity-lora
files.download("contract-clarity-lora.zip")